# 🧹 Data Cleaning Notebook

---

## What is this notebook about?

**In simple terms:** This notebook takes raw (messy) data about food prices around the world and cleans it up so we can analyse it properly.

Think of it like preparing ingredients before cooking - we need to wash, chop, and organise everything before we can make a proper meal!

---

## What will we do?

1. **Load the data** - Read the original file into our notebook
2. **Explore the data** - Take a first look at what we're working with
3. **Find problems** - Check for missing information and errors
4. **Fix problems** - Clean up any issues we find
5. **Create helpful new columns** - Add useful calculations
6. **Save the clean data** - Store our cleaned data for later use

---

## About the Dataset

| Item | Details |
|------|---------|
| **What is it?** | Food price data from countries around the world |
| **Where from?** | World Bank - Real-Time Food Prices |
| **Time period** | January 2007 to October 2023 (over 16 years!) |
| **Original file** | `data/raw/WLD_RTFP_country_2023-10-02.csv` |
| **Cleaned file** | `data/cleaned/food_prices_cleaned.csv` |

---
## Step 1: Import the Tools We Need

**What's happening?** We're loading special Python tools (called "libraries") that help us work with data.

Think of these like kitchen appliances - each one has a specific job:
- **pandas** - Our main tool for handling data (like a food processor)
- **numpy** - Helps with mathematical calculations (like a calculator)
- **os** - Helps us work with files and folders (like a filing cabinet)

In [ ]:
# Import our data tools
import pandas as pd    # For working with tables of data
import numpy as np     # For mathematical operations
import os              # For working with files and folders

print("✅ All tools loaded successfully!")

---
## Step 2: Load the Raw Data

**What's happening?** We're reading the original data file into our notebook.

This is like opening a spreadsheet - the data comes in as a table with rows (each observation) and columns (different types of information).

In [ ]:
# Tell Python where to find our data file
raw_data_path = '../data/raw/WLD_RTFP_country_2023-10-02.csv'

# Read the file into a "dataframe" (a table of data)
df = pd.read_csv(raw_data_path)

# Show what we loaded
print("✅ Data loaded successfully!")
print(f"")
print(f"📊 Dataset size:")
print(f"   • {df.shape[0]:,} rows (individual records)")
print(f"   • {df.shape[1]} columns (types of information)")

---
## Step 3: First Look at Our Data

**What's happening?** We're exploring what's in our dataset before we start cleaning.

This is like doing an inventory check - we need to know what we have before we can organise it!

### 3.1 Preview the First 10 Rows

Let's see what the data looks like:

In [ ]:
# Show the first 10 rows of our data
df.head(10)

### 💡 What do these columns mean?

| Column | Plain English Meaning |
|--------|----------------------|
| **Open** | The food price at the start of the month |
| **High** | The highest food price during that month |
| **Low** | The lowest food price during that month |
| **Close** | The food price at the end of the month |
| **Inflation** | How much prices changed compared to last year (as a percentage) |
| **country** | The country name |
| **ISO3** | A 3-letter code for the country (e.g., AFG = Afghanistan) |
| **date** | The month and year of the measurement |

### 3.2 Check Data Types

**What's happening?** We're checking what type of information each column contains (numbers, text, dates, etc.)

In [ ]:
# Show information about each column
print("📋 Column Information:")
print("="*50)
df.info()

### 3.3 Statistical Summary

**What's happening?** We're calculating basic statistics (average, minimum, maximum, etc.) for our numerical columns.

In [ ]:
# Show basic statistics for numerical columns
df.describe()

### 3.4 Which Countries Are in Our Data?

Let's see all the countries we have data for:

In [ ]:
# Count unique countries
num_countries = df['country'].nunique()
countries_list = df['country'].unique()

print(f"🌍 We have data for {num_countries} different countries:")
print("="*50)
for i, country in enumerate(sorted(countries_list), 1):
    print(f"   {i}. {country}")

---
## Step 4: Find Missing Values

**What's happening?** We're checking if any information is missing from our data.

Missing values are like empty cells in a spreadsheet - sometimes data wasn't recorded or got lost. We need to know about these gaps!

In [ ]:
# Count missing values for each column
missing_values = df.isnull().sum()
missing_percentage = (df.isnull().sum() / len(df)) * 100

# Create a summary table
missing_df = pd.DataFrame({
    'Missing Values': missing_values,
    'Percentage (%)': missing_percentage.round(2)
})

print("🔍 Missing Values Report")
print("="*50)
print(missing_df)
print("="*50)

# Highlight the key finding
if missing_values.sum() > 0:
    print(f"\n⚠️ We found {missing_values.sum()} missing values in total!")
    print(f"   Most missing values are in the 'Inflation' column.")
else:
    print("\n✅ No missing values found!")

### 💡 Why is inflation data missing for some records?

**Simple explanation:** The inflation column shows how much prices changed compared to the same month last year. 

For the first year of data (2007), we can't calculate this because there's no previous year to compare with! That's why those early records have no inflation value - it's not an error, it's just unavailable.

---
## Step 5: Check for Duplicate Records

**What's happening?** We're checking if any records appear more than once.

Duplicates are like having the same entry written twice - they can mess up our calculations if we don't remove them.

In [ ]:
# Check for exact duplicate rows
duplicates = df.duplicated().sum()
print(f"🔄 Duplicate Records Check")
print("="*50)
print(f"   Exact duplicate rows found: {duplicates}")

# Check if any country has two entries for the same date
duplicate_country_date = df.duplicated(subset=['country', 'date']).sum()
print(f"   Same country + same date duplicates: {duplicate_country_date}")

if duplicates == 0 and duplicate_country_date == 0:
    print("\n✅ Great news! No duplicates found - our data is clean in this regard.")
else:
    print(f"\n⚠️ Found duplicates that need to be removed!")

---
## Step 6: Clean the Data

Now comes the actual cleaning! We'll fix issues and improve our data structure.

### 6.1 Fix the Date Column

**What's happening?** Right now, dates are stored as text (like "2007-01-01"). We need to convert them to proper dates so Python can work with them correctly.

We'll also extract the year and month separately - this will be useful for our analysis later!

In [ ]:
# Convert the date column from text to actual dates
df['date'] = pd.to_datetime(df['date'])

# Extract year and month into separate columns
df['year'] = df['date'].dt.year
df['month'] = df['date'].dt.month
df['month_name'] = df['date'].dt.month_name()

# Show the date range
print("📅 Date Conversion Complete!")
print("="*50)
print(f"   Earliest date: {df['date'].min().strftime('%B %Y')}")
print(f"   Latest date:   {df['date'].max().strftime('%B %Y')}")
print(f"   Time span:     {(df['date'].max() - df['date'].min()).days // 365} years")
print("\n✅ New columns added: year, month, month_name")

### 6.2 Handle Missing Inflation Values

**What's happening?** We found that some inflation values are missing. Rather than delete these records or make up numbers, we'll:

1. Keep the missing values as they are
2. Add a new column that tells us whether inflation data is available

This way, we can easily filter our analysis later to only include records with complete data.

In [ ]:
# Create a flag column: True if inflation data exists, False if missing
df['inflation_available'] = df['Inflation'].notna()

# Count the records
with_inflation = df['inflation_available'].sum()
without_inflation = (~df['inflation_available']).sum()

print("📊 Inflation Data Availability")
print("="*50)
print(f"   ✅ Records WITH inflation data:    {with_inflation:,} ({with_inflation/len(df)*100:.1f}%)")
print(f"   ⚠️ Records WITHOUT inflation data: {without_inflation:,} ({without_inflation/len(df)*100:.1f}%)")
print("\n💡 Tip: Records without inflation are mostly from the first year (2007)")
print("         because we need a previous year to calculate year-over-year change.")

### 6.3 Remove Any Duplicate Records

**What's happening?** Even though we didn't find duplicates earlier, it's good practice to run the removal step anyway - just to be safe!

In [ ]:
# Record how many rows we have before
rows_before = len(df)

# Remove any duplicate rows
df = df.drop_duplicates()

# Count how many were removed
rows_removed = rows_before - len(df)

print("🗑️ Duplicate Removal")
print("="*50)
print(f"   Rows before: {rows_before:,}")
print(f"   Rows after:  {len(df):,}")
print(f"   Removed:     {rows_removed}")

if rows_removed == 0:
    print("\n✅ No duplicates to remove - data was already clean!")

### 6.4 Validate the Data Makes Sense

**What's happening?** We're doing some logical checks to make sure our data is reasonable:

1. **Prices should not be negative** - You can't have a negative food price!
2. **High price should be >= Low price** - The highest price can't be lower than the lowest price

In [ ]:
print("🔍 Data Validation Checks")
print("="*50)

# Check 1: No negative prices
print("\nCheck 1: Are there any negative prices?")
price_columns = ['Open', 'High', 'Low', 'Close']

all_positive = True
for col in price_columns:
    negative_count = (df[col] < 0).sum()
    if negative_count > 0:
        print(f"   ❌ {col}: Found {negative_count} negative values")
        all_positive = False
    else:
        print(f"   ✅ {col}: All values are positive")

# Check 2: High >= Low always
print("\nCheck 2: Is 'High' always >= 'Low'?")
inconsistent = (df['High'] < df['Low']).sum()
if inconsistent > 0:
    print(f"   ❌ Found {inconsistent} rows where High < Low")
else:
    print(f"   ✅ All rows have High >= Low (as expected)")

print("\n" + "="*50)
if all_positive and inconsistent == 0:
    print("✅ All validation checks PASSED!")
else:
    print("⚠️ Some validation checks FAILED - review needed!")

### 6.5 Create Useful New Columns

**What's happening?** We're calculating some new values that will be helpful for our analysis:

| New Column | What it shows | Why it's useful |
|------------|---------------|------------------|
| **price_range** | High minus Low | Shows how much prices jumped around (volatility) |
| **price_change** | Close minus Open | Shows if prices went up or down during the month |
| **price_change_pct** | Same as above, but as a percentage | Easier to compare across different price levels |

In [ ]:
# Calculate price range (how much prices varied during the month)
df['price_range'] = df['High'] - df['Low']

# Calculate price change (did prices go up or down?)
df['price_change'] = df['Close'] - df['Open']

# Calculate percentage change (easier to interpret)
df['price_change_pct'] = ((df['Close'] - df['Open']) / df['Open']) * 100

print("✨ New Columns Created!")
print("="*50)
print("\n1️⃣ price_range = High - Low")
print("   → Tells us how volatile (jumpy) prices were")
print(f"   → Average: {df['price_range'].mean():.4f}")

print("\n2️⃣ price_change = Close - Open")
print("   → Positive = prices went UP, Negative = prices went DOWN")
print(f"   → Average: {df['price_change'].mean():.4f}")

print("\n3️⃣ price_change_pct = Same as above, but as %")
print(f"   → Average monthly change: {df['price_change_pct'].mean():.2f}%")

### 6.6 Standardise Column Names

**What's happening?** We're renaming columns to use a consistent format (all lowercase with underscores). This is a best practice that makes the data easier to work with.

In [ ]:
# Rename columns to lowercase with underscores (called "snake_case")
df = df.rename(columns={
    'Open': 'open',
    'High': 'high',
    'Low': 'low',
    'Close': 'close',
    'Inflation': 'inflation',
    'ISO3': 'iso3'
})

print("📝 Column Names Standardised")
print("="*50)
print("\nBefore → After:")
print("   Open → open")
print("   High → high")
print("   Low → low")
print("   Close → close")
print("   Inflation → inflation")
print("   ISO3 → iso3")
print("\n✅ All column names are now in lowercase!")

### 6.7 Organise Columns in a Logical Order

**What's happening?** We're rearranging the columns so related information is grouped together. This makes the data easier to read and understand.

In [ ]:
# Define the new column order
column_order = [
    # Date information first
    'date', 'year', 'month', 'month_name',
    # Then location
    'country', 'iso3',
    # Then price data
    'open', 'high', 'low', 'close',
    # Then calculated features
    'price_range', 'price_change', 'price_change_pct',
    # Finally, inflation data
    'inflation', 'inflation_available'
]

# Reorder the columns
df = df[column_order]

print("📋 Columns Reordered Logically")
print("="*50)
print("\nColumn Groups:")
print("   📅 Date info:     date, year, month, month_name")
print("   🌍 Location:      country, iso3")
print("   💰 Price data:    open, high, low, close")
print("   📊 Calculations:  price_range, price_change, price_change_pct")
print("   📈 Inflation:     inflation, inflation_available")

---
## Step 7: Review the Cleaned Data

**What's happening?** Let's take a final look at our cleaned data to make sure everything looks good!

In [ ]:
print("✅ FINAL CLEANED DATA SUMMARY")
print("="*60)

print(f"\n📊 Dataset Size:")
print(f"   • {df.shape[0]:,} rows (records)")
print(f"   • {df.shape[1]} columns")

print(f"\n📅 Time Period:")
print(f"   • From: {df['date'].min().strftime('%B %Y')}")
print(f"   • To:   {df['date'].max().strftime('%B %Y')}")

print(f"\n🌍 Geographic Coverage:")
print(f"   • {df['country'].nunique()} countries")

print(f"\n⚠️ Missing Values:")
missing = df.isnull().sum()
if missing.sum() > 0:
    for col, count in missing[missing > 0].items():
        print(f"   • {col}: {count:,} missing ({count/len(df)*100:.1f}%)")
else:
    print("   • No missing values!")

print("\n" + "="*60)

In [ ]:
# Show the first 10 rows of clean data
print("👀 Preview of Cleaned Data (first 10 rows):")
df.head(10)

In [ ]:
# Show data types
print("📋 Data Types for Each Column:")
print(df.dtypes)

---
## Step 8: Save the Cleaned Data

**What's happening?** We're saving our cleaned data to a new file. This way:
- The original (raw) data stays untouched
- We can use the clean data for our analysis
- We don't need to run the cleaning steps again

In [ ]:
# Create the output folder if it doesn't exist
output_dir = '../data/cleaned'
os.makedirs(output_dir, exist_ok=True)

# Save the cleaned data
output_path = os.path.join(output_dir, 'food_prices_cleaned.csv')
df.to_csv(output_path, index=False)

# Confirm it saved correctly
file_size = os.path.getsize(output_path) / 1024  # Convert to KB

print("💾 Data Saved Successfully!")
print("="*50)
print(f"   📁 Location: {output_path}")
print(f"   📏 File size: {file_size:.1f} KB")
print(f"   📊 Contains: {len(df):,} rows × {len(df.columns)} columns")
print("\n✅ Your clean data is ready for analysis!")

---
## 📝 Summary: What We Did

### The Cleaning Steps We Performed:

| Step | What We Did | Why It Matters |
|------|-------------|----------------|
| 1 | Loaded the raw data | Got our data into Python |
| 2 | Explored the structure | Understood what we're working with |
| 3 | Checked for missing values | Found 364 missing inflation values (7.6%) |
| 4 | Checked for duplicates | Confirmed no duplicate records |
| 5 | Converted dates | Made dates work properly in Python |
| 6 | Added date columns | Created year, month, month_name for analysis |
| 7 | Flagged missing inflation | Created a helper column to track data availability |
| 8 | Validated the data | Confirmed no negative prices or logical errors |
| 9 | Created new features | Added price_range, price_change, price_change_pct |
| 10 | Standardised column names | Made everything lowercase and consistent |
| 11 | Organised columns | Grouped related information together |
| 12 | Saved the clean data | Stored results for future analysis |

---

### Key Outcomes:

✅ **4,798 records** spanning **16+ years** (2007-2023)  
✅ **24 countries** with food price data  
✅ **15 clean columns** ready for analysis  
✅ Clean data saved to `data/cleaned/food_prices_cleaned.csv`

---

### Next Steps:

➡️ Open the **Data_Analysis.ipynb** notebook to explore patterns and trends  
➡️ Open the **Hypothesis_Testing.ipynb** notebook to test our theories statistically

---
## 🤖 AI Assistance Note

This notebook was created with assistance from **GitHub Copilot** (an AI coding assistant). The AI helped with:
- Writing code for data cleaning tasks
- Suggesting best practices for data validation
- Creating clear explanations and documentation

All code was reviewed and tested by the analyst to ensure accuracy.

# Data Cleaning Notebook

## Objective
Clean and preprocess the World Real-Time Food Prices dataset for further analysis.

## Dataset
- **Source**: `data/raw/WLD_RTFP_country_2023-10-02.csv`
- **Output**: `data/cleaned/food_prices_cleaned.csv`

## 1. Import Libraries

In [10]:
import pandas as pd
import numpy as np
import os

## 2. Load Raw Data

In [11]:
# Load the raw dataset
raw_data_path = '../data/raw/WLD_RTFP_country_2023-10-02.csv'
df = pd.read_csv(raw_data_path)

print(f"Dataset loaded successfully!")
print(f"Shape: {df.shape[0]} rows, {df.shape[1]} columns")

Dataset loaded successfully!
Shape: 4798 rows, 8 columns


## 3. Initial Data Exploration

In [12]:
# Display first few rows
df.head(10)

,Open,High,Low,Close,Inflation,country,ISO3,date
0,0.53,0.54,0.53,0.53,NaN,Afghanistan,AFG,2007-01-01
1,0.53,0.54,0.53,0.53,NaN,Afghanistan,AFG,2007-02-01
2,0.54,0.54,0.53,0.53,NaN,Afghanistan,AFG,2007-03-01
3,0.53,0.55,0.53,0.55,NaN,Afghanistan,AFG,2007-04-01
4,0.56,0.57,0.56,0.57,NaN,Afghanistan,AFG,2007-05-01
5,0.59,0.60,0.58,0.58,NaN,Afghanistan,AFG,2007-06-01
6,0.59,0.60,0.59,0.59,NaN,Afghanistan,AFG,2007-07-01
7,0.60,0.60,0.59,0.59,NaN,Afghanistan,AFG,2007-08-01
8,0.60,0.62,0.59,0.62,NaN,Afghanistan,AFG,2007-09-01
9,0.64,0.64,0.63,0.63,NaN,Afghanistan,AFG,2007-10-01


In [13]:
# Data types and info
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 4798 entries, 0 to 4797
Data columns (total 8 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   Open       4734 non-null   float64
 1   High       4734 non-null   float64
 2   Low        4734 non-null   float64
 3   Close      4734 non-null   float64
 4   Inflation  4434 non-null   float64
 5   country    4798 non-null   str    
 6   ISO3       4798 non-null   str    
 7   date       4798 non-null   str    
dtypes: float64(5), str(3)
memory usage: 300.0 KB


In [14]:
# Statistical summary
df.describe()

,Open,High,Low,Close,Inflation
count,4734.000000,4734.000000,4734.000000,4734.000000,4434.000000
mean,1.491880,1.536158,1.451056,1.492398,14.692346
std,4.652457,4.883312,4.439229,4.633321,35.910342
min,0.010000,0.010000,0.010000,0.010000,-31.470000
25%,0.740000,0.750000,0.720000,0.740000,-0.487500
50%,0.960000,0.980000,0.950000,0.960000,5.360000
75%,1.100000,1.120000,1.077500,1.100000,16.372500
max,102.460000,106.480000,94.420000,94.420000,363.100000


In [15]:
# Check unique countries
print(f"Number of unique countries: {df['country'].nunique()}")
print(f"\nCountries: {df['country'].unique()}")

Number of unique countries: 25

Countries: <StringArray>
[             'Afghanistan',                  'Burundi',
             'Burkina Faso', 'Central African Republic',
                 'Cameroon',         'Congo, Dem. Rep.',
              'Congo, Rep.',              'Gambia, The',
            'Guinea-Bissau',                    'Haiti',
                     'Iraq',                  'Lao PDR',
                  'Lebanon',                  'Liberia',
                     'Mali',                  'Myanmar',
               'Mozambique',                    'Niger',
                  'Nigeria',                    'Sudan',
                  'Somalia',              'South Sudan',
     'Syrian Arab Republic',                     'Chad',
              'Yemen, Rep.']
Length: 25, dtype: str


## 4. Check for Missing Values

In [16]:
# Count missing values per column
missing_values = df.isnull().sum()
missing_percentage = (df.isnull().sum() / len(df)) * 100

missing_df = pd.DataFrame({
    'Missing Values': missing_values,
    'Percentage (%)': missing_percentage.round(2)
})

print("Missing Values Analysis:")
missing_df

Missing Values Analysis:


,Missing Values,Percentage (%)
Open,64,1.33
High,64,1.33
Low,64,1.33
Close,64,1.33
Inflation,364,7.59
country,0,0.00
ISO3,0,0.00
date,0,0.00


## 5. Check for Duplicates

In [17]:
# Check for duplicate rows
duplicates = df.duplicated().sum()
print(f"Number of duplicate rows: {duplicates}")

# Check for duplicate country-date combinations
duplicate_country_date = df.duplicated(subset=['country', 'date']).sum()
print(f"Duplicate country-date combinations: {duplicate_country_date}")

Number of duplicate rows: 0
Duplicate country-date combinations: 0


## 6. Data Cleaning Steps

### 6.1 Convert Date Column to DateTime

In [18]:
# Convert date column to datetime
df['date'] = pd.to_datetime(df['date'])

# Extract useful date components
df['year'] = df['date'].dt.year
df['month'] = df['date'].dt.month
df['month_name'] = df['date'].dt.month_name()

print("Date range:")
print(f"From: {df['date'].min()}")
print(f"To: {df['date'].max()}")

Date range:
From: 2007-01-01 00:00:00
To: 2023-10-01 00:00:00


### 6.2 Handle Missing Values in Inflation Column

In [19]:
# Check missing inflation values by country
missing_inflation_by_country = df[df['Inflation'].isnull()].groupby('country').size()
print("Missing Inflation values by country:")
print(missing_inflation_by_country)

Missing Inflation values by country:
country
Afghanistan                 12
Burkina Faso                12
Burundi                     12
Cameroon                    51
Central African Republic    12
Chad                        12
Congo, Dem. Rep.            12
Congo, Rep.                 12
Gambia, The                 12
Guinea-Bissau               12
Haiti                       12
Iraq                        12
Lao PDR                     37
Lebanon                     12
Liberia                     12
Mali                        12
Mozambique                  12
Myanmar                     12
Niger                       12
Nigeria                     12
Somalia                     12
South Sudan                 12
Sudan                       12
Syrian Arab Republic        12
Yemen, Rep.                 12
dtype: int64


In [20]:
# Option 1: Keep missing values as NaN (for analysis where we handle missing data)
# Option 2: Fill missing inflation with 0 (if inflation data not available)
# Option 3: Forward fill within each country group

# We'll create a cleaned version with NaN preserved and add a flag column
df['inflation_available'] = df['Inflation'].notna()

print(f"Records with inflation data: {df['inflation_available'].sum()}")
print(f"Records without inflation data: {(~df['inflation_available']).sum()}")

Records with inflation data: 4434
Records without inflation data: 364


### 6.3 Remove Duplicate Rows (if any)

In [21]:
# Remove exact duplicate rows
initial_rows = len(df)
df = df.drop_duplicates()
rows_removed = initial_rows - len(df)
print(f"Duplicate rows removed: {rows_removed}")

Duplicate rows removed: 0


### 6.4 Validate Numeric Columns

In [22]:
# Check for negative values in price columns (Open, High, Low, Close)
price_columns = ['Open', 'High', 'Low', 'Close']

for col in price_columns:
    negative_count = (df[col] < 0).sum()
    if negative_count > 0:
        print(f"Warning: {col} has {negative_count} negative values")
    else:
        print(f"{col}: No negative values (OK)")

Open: No negative values (OK)
High: No negative values (OK)
Low: No negative values (OK)
Close: No negative values (OK)


In [23]:
# Check data consistency: High should be >= Low
inconsistent_high_low = df[df['High'] < df['Low']]
print(f"Rows where High < Low: {len(inconsistent_high_low)}")

if len(inconsistent_high_low) > 0:
    print("Inconsistent rows:")
    display(inconsistent_high_low)

Rows where High < Low: 0


### 6.5 Calculate Additional Features

In [24]:
# Calculate price range (volatility indicator)
df['price_range'] = df['High'] - df['Low']

# Calculate price change (Close - Open)
df['price_change'] = df['Close'] - df['Open']

# Calculate percentage change
df['price_change_pct'] = ((df['Close'] - df['Open']) / df['Open']) * 100

print("New features added: price_range, price_change, price_change_pct")

New features added: price_range, price_change, price_change_pct


### 6.6 Rename Columns for Consistency

In [25]:
# Rename columns to snake_case for consistency
df = df.rename(columns={
    'Open': 'open',
    'High': 'high',
    'Low': 'low',
    'Close': 'close',
    'Inflation': 'inflation',
    'ISO3': 'iso3'
})

print("Columns renamed to snake_case")
print(f"Current columns: {df.columns.tolist()}")

Columns renamed to snake_case
Current columns: ['open', 'high', 'low', 'close', 'inflation', 'country', 'iso3', 'date', 'year', 'month', 'month_name', 'inflation_available', 'price_range', 'price_change', 'price_change_pct']


### 6.7 Reorder Columns

In [26]:
# Reorder columns logically
column_order = [
    'date', 'year', 'month', 'month_name',
    'country', 'iso3',
    'open', 'high', 'low', 'close',
    'price_range', 'price_change', 'price_change_pct',
    'inflation', 'inflation_available'
]

df = df[column_order]
print("Columns reordered")

Columns reordered


## 7. Final Data Validation

In [27]:
# Final check of cleaned data
print("=" * 50)
print("CLEANED DATA SUMMARY")
print("=" * 50)
print(f"\nShape: {df.shape[0]} rows, {df.shape[1]} columns")
print(f"\nDate Range: {df['date'].min()} to {df['date'].max()}")
print(f"\nCountries: {df['country'].nunique()}")
print(f"\nMissing Values:")
print(df.isnull().sum())

CLEANED DATA SUMMARY

Shape: 4798 rows, 15 columns

Date Range: 2007-01-01 00:00:00 to 2023-10-01 00:00:00

Countries: 25

Missing Values:
date                     0
year                     0
month                    0
month_name               0
country                  0
iso3                     0
open                    64
high                    64
low                     64
close                   64
price_range             64
price_change            64
price_change_pct        64
inflation              364
inflation_available      0
dtype: int64


In [28]:
# Preview cleaned data
df.head(10)

,date,year,month,month_name,country,iso3,open,high,low,close,price_range,price_change,price_change_pct,inflation,inflation_available
0,2007-01-01,2007,1,January,Afghanistan,AFG,0.53,0.54,0.53,0.53,0.01,0.00,0.000000,NaN,False
1,2007-02-01,2007,2,February,Afghanistan,AFG,0.53,0.54,0.53,0.53,0.01,0.00,0.000000,NaN,False
2,2007-03-01,2007,3,March,Afghanistan,AFG,0.54,0.54,0.53,0.53,0.01,-0.01,-1.851852,NaN,False
3,2007-04-01,2007,4,April,Afghanistan,AFG,0.53,0.55,0.53,0.55,0.02,0.02,3.773585,NaN,False
4,2007-05-01,2007,5,May,Afghanistan,AFG,0.56,0.57,0.56,0.57,0.01,0.01,1.785714,NaN,False
5,2007-06-01,2007,6,June,Afghanistan,AFG,0.59,0.60,0.58,0.58,0.02,-0.01,-1.694915,NaN,False
6,2007-07-01,2007,7,July,Afghanistan,AFG,0.59,0.60,0.59,0.59,0.01,0.00,0.000000,NaN,False
7,2007-08-01,2007,8,August,Afghanistan,AFG,0.60,0.60,0.59,0.59,0.01,-0.01,-1.666667,NaN,False
8,2007-09-01,2007,9,September,Afghanistan,AFG,0.60,0.62,0.59,0.62,0.03,0.02,3.333333,NaN,False
9,2007-10-01,2007,10,October,Afghanistan,AFG,0.64,0.64,0.63,0.63,0.01,-0.01,-1.562500,NaN,False


In [29]:
# Data types of cleaned dataset
df.dtypes

date                   datetime64[us]
year                            int32
month                           int32
month_name                        str
country                           str
iso3                              str
open                          float64
high                          float64
low                           float64
close                         float64
price_range                   float64
price_change                  float64
price_change_pct              float64
inflation                     float64
inflation_available              bool
dtype: object

## 8. Save Cleaned Data

In [30]:
# Create output directory if it doesn't exist
output_dir = '../data/cleaned'
os.makedirs(output_dir, exist_ok=True)

# Save cleaned data to CSV
output_path = os.path.join(output_dir, 'food_prices_cleaned.csv')
df.to_csv(output_path, index=False)

print(f"Cleaned data saved to: {output_path}")
print(f"File size: {os.path.getsize(output_path) / 1024:.2f} KB")

Cleaned data saved to: ../data/cleaned\food_prices_cleaned.csv
File size: 565.51 KB


## 9. Summary of Cleaning Steps

The following cleaning operations were performed:

1. **Loaded raw data** from CSV file
2. **Converted date column** to datetime format
3. **Extracted date components** (year, month, month_name)
4. **Handled missing values** - added flag for missing inflation data
5. **Removed duplicates** (if any)
6. **Validated numeric data** - checked for negative prices and data consistency
7. **Created new features** - price_range, price_change, price_change_pct
8. **Renamed columns** to snake_case for consistency
9. **Reordered columns** logically
10. **Saved cleaned data** to `data/cleaned/food_prices_cleaned.csv`